# 📊 Notebook 01 — Ranking Functions

**Functions covered:** `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `NTILE()`, `PERCENT_RANK()`, `CUME_DIST()`

## Quick Reference

| Function | Ties | Gaps | Use when... |
|----------|------|------|-------------|
| `ROW_NUMBER()` | unique int per row | — | You need a unique sequential ID |
| `RANK()` | same rank | yes | Official leaderboard (Olympic style) |
| `DENSE_RANK()` | same rank | no | No skipped positions |
| `NTILE(n)` | splits evenly | — | Bucket/quartile assignment |
| `PERCENT_RANK()` | 0.0–1.0 | — | Relative standing as a percentage |
| `CUME_DIST()` | 0.0–1.0 | — | Cumulative distribution |


In [ ]:
import duckdb, pandas as pd

employees = pd.read_csv("../data/employees.csv")
sales     = pd.read_csv("../data/sales.csv")
con = duckdb.connect()
con.register("employees", employees)
con.register("sales",     sales)

def sql(query):
    return con.execute(query).fetchdf()


---
## Exercise 1 · The Three Ranking Functions

**Question:** List every employee's `full_name`, `department`, `salary`, and their salary rank within the company using **all three** ranking functions (`ROW_NUMBER`, `RANK`, `DENSE_RANK`), ordered by rank ascending.

*Tip: Which employees share a rank? How do the results differ?*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    full_name,
    department,
    salary,
    ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_num,
    RANK()       OVER (ORDER BY salary DESC) AS rnk,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS dense_rnk
FROM employees
ORDER BY rnk;
```
</details>

---
## Exercise 2 · Top 3 Salaries Per Department

**Question:** Find the **top 3 highest-paid employees in each department**. Include ties. Show `full_name`, `department`, `salary`, and `dept_rank`.

*Tip: Use `DENSE_RANK()` so ties don't exclude employees.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT *
FROM (
    SELECT
        full_name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
    FROM employees
)
WHERE dept_rank <= 3
ORDER BY department, dept_rank;
```
</details>

---
## Exercise 3 · Second Highest Salary Per Department

**Question:** Find the employee(s) with the **2nd highest unique salary** in each department. Show `full_name`, `department`, `salary`.

*Tip: When using `DENSE_RANK`, the 2nd highest unique salary has rank = 2.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT full_name, department, salary
FROM (
    SELECT
        full_name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank
    FROM employees
)
WHERE dept_rank = 2
ORDER BY department;
```
</details>

---
## Exercise 4 · Salary Quartiles with NTILE

**Question:** Divide all employees into **4 salary quartiles** (1 = lowest, 4 = highest) using `NTILE(4)`. Show `full_name`, `salary`, and `salary_quartile`, sorted by quartile and salary.

*Bonus: How many employees fall into each quartile?*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
-- Main query
SELECT
    full_name,
    salary,
    NTILE(4) OVER (ORDER BY salary) AS salary_quartile
FROM employees
ORDER BY salary_quartile, salary;

-- Bonus: distribution count
-- SELECT salary_quartile, COUNT(*) AS cnt
-- FROM (
--     SELECT NTILE(4) OVER (ORDER BY salary) AS salary_quartile FROM employees
-- )
-- GROUP BY 1 ORDER BY 1;
```
</details>

---
## Exercise 5 · Top 10% Earners (PERCENT_RANK)

**Question:** Identify all employees who are in the **top 10% of salary** company-wide. Show `full_name`, `department`, `salary`, and `pct_rank` (as a percentage rounded to 1 decimal).

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT full_name, department, salary, pct_rank
FROM (
    SELECT
        full_name,
        department,
        salary,
        ROUND(PERCENT_RANK() OVER (ORDER BY salary) * 100, 1) AS pct_rank
    FROM employees
)
WHERE pct_rank >= 90
ORDER BY salary DESC;
```
</details>

---
## Exercise 6 · Rank Sales Reps by Quarterly Revenue

**Question:** Rank each sales rep by their **total revenue in Q4 2023** within their region. Show `region`, `rep_name`, `total_q4_revenue`, and `region_rank`. Only include Q4 2023 data.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    region,
    rep_name,
    total_q4_revenue,
    RANK() OVER (PARTITION BY region ORDER BY total_q4_revenue DESC) AS region_rank
FROM (
    SELECT
        region,
        rep_name,
        SUM(amount) AS total_q4_revenue
    FROM sales
    WHERE quarter = 'Q4' AND year = 2023
    GROUP BY region, rep_name
)
ORDER BY region, region_rank;
```
</details>

---
## Exercise 7 · Detect Duplicate Salaries

**Question:** Find all departments where **at least two employees have the exact same salary**. For each such salary, list the department, salary value, and all employee names sharing it.

*Tip: Use `COUNT()` as a window function to count peers with the same salary.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT department, salary, full_name
FROM (
    SELECT
        department,
        salary,
        full_name,
        COUNT(*) OVER (PARTITION BY department, salary) AS salary_count
    FROM employees
)
WHERE salary_count > 1
ORDER BY department, salary;
```
</details>

---
## Exercise 8 · Number Each Rep's Sales Chronologically

**Question:** For each sales rep, assign a sequential order number (`sale_number`) to their sales, starting from 1 for their earliest sale. Show `rep_name`, `sale_date`, `amount`, and `sale_number`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    ROW_NUMBER() OVER (PARTITION BY rep_name ORDER BY sale_date) AS sale_number
FROM sales
ORDER BY rep_name, sale_number;
```
</details>

---
## 🎯 Summary

- `ROW_NUMBER()` → always unique; use for pagination or deduplication.
- `RANK()` → ties get same rank, next rank skips (1, 2, 2, 4).
- `DENSE_RANK()` → ties get same rank, no skipping (1, 2, 2, 3) — best for "top-N per group".
- `NTILE(n)` → distributes rows into n buckets; great for quartile/decile analysis.
- `PERCENT_RANK()` / `CUME_DIST()` → relative positioning (0.0–1.0).

➡️ **Next:** `02_offset_functions.ipynb` — LAG, LEAD, FIRST_VALUE, LAST_VALUE